# Phase 4 - Validation EDA

## tl;dr

This notebook audits whether the current local validation setup is trustworthy enough for the next modeling loop. It compares current sorted grouped folds, a descriptor-balanced fold assignment, endpoint/slope baselines, historical model fold metrics, and public leaderboard gaps.

## Context & Methods

Local validation should simulate the competition setting: predict the trailing target zone after `TVT_input` becomes missing. Phase 4 uses per-well descriptors from Phase 2, target-zone baseline errors computed directly from raw train files, and prior model fold artifacts.

### Key Assumptions

- Public leaderboard scores are external checks, not a validation set.
- Visible local test wells are schema/submission examples only because Phase 2 found they overlap train.
- Fold diagnostics should be assessed at both row-weighted and well-level grains because wells vary substantially in target-zone row counts.


In [1]:
"Configure imports, paths, and deterministic parameters."

from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "eda_phase4_validation"
RUN_DATE = "2026-06-21"
SEED = 42
N_SPLITS = 3

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")


In [2]:
"Load descriptor and prior-result artifacts."

train_summary = pd.read_csv(TABLE_DIR / f"eda__phase2_train_well_summary__{RUN_DATE}.csv")
typewell_alignment = pd.read_csv(TABLE_DIR / f"eda__phase2_typewell_alignment__{RUN_DATE}.csv")
model_comparison = pd.read_csv(TABLE_DIR / "model_comparison.csv")
leaderboard = pd.DataFrame([
    {"model_name": "ridge_alpha_10", "public_lb_rmse": 15.905, "submission_ref": "53923555"},
    {"model_name": "random_forest", "public_lb_rmse": 15.873, "submission_ref": "53924036"},
    {"model_name": "elasticnet_alpha_0p001_l1_0p2", "public_lb_rmse": 16.091, "submission_ref": "53924934"},
])
well_descriptors = train_summary.merge(typewell_alignment, on="well", how="left")

descriptor_columns = [
    "eval_frac", "eval_rows", "md_range", "eval_md_range", "gr_missing_frac", "gr_eval_missing_frac",
    "last_known_tvt_input", "known_tvt_range", "known_tvt_slope_last_50", "eval_tvt_delta_from_last_known",
    "horizontal_length", "horiz_typewell_gr_corr", "horiz_typewell_gr_absdiff_median",
]
for column in descriptor_columns:
    well_descriptors[column] = pd.to_numeric(well_descriptors[column], errors="coerce")

print({
    "wells": len(well_descriptors),
    "descriptor_columns": len(descriptor_columns),
    "model_rows": len(model_comparison),
})
well_descriptors[["well"] + descriptor_columns].head()


{'wells': 773, 'descriptor_columns': 13, 'model_rows': 6}


,well,eval_frac,eval_rows,md_range,eval_md_range,gr_missing_frac,gr_eval_missing_frac,last_known_tvt_input,known_tvt_range,known_tvt_slope_last_50,eval_tvt_delta_from_last_known,horizontal_length,horiz_typewell_gr_corr,horiz_typewell_gr_absdiff_median
0,000d7d20,0.726790,3836,5277.0,3835.0,0.427814,0.473410,11747.37,520.10,0.02,-10.06,5028.530211,0.784472,6.995662
1,00bbac68,0.795608,6014,7558.0,6013.0,0.124620,0.138344,12223.54,818.42,0.01,-6.74,7146.672332,0.823135,7.031787
2,00e12e8b,0.673716,4301,6383.0,4300.0,0.091479,0.097187,11604.82,998.64,0.04,-8.45,5753.432580,0.931801,11.624651
3,015fe0d2,0.722017,4296,5949.0,4295.0,0.254286,0.320298,12233.78,493.93,0.03,23.98,5710.270740,0.863160,10.719077
4,01869cd4,0.811241,5557,6849.0,5556.0,0.477080,0.511247,11745.94,387.67,0.01,3.45,6655.423635,0.785929,7.182613


In [3]:
"Compute endpoint-hold and slope-projection baselines per train well."


def rmse_value(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def per_well_target_baselines(well):
    horizontal = load_horizontal_well(TRAIN_DIR / f"{well}__horizontal_well.csv")
    target = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target.sum()) == 0 or int(known.sum()) == 0:
        return None
    known_pos = np.flatnonzero(known.to_numpy())
    last_pos = int(known_pos[-1])
    last_tvt = float(horizontal["TVT_input"].iloc[last_pos])
    last_md = float(horizontal["MD"].iloc[last_pos])
    y_true = horizontal.loc[target, "TVT"].to_numpy(float)
    md_eval = horizontal.loc[target, "MD"].to_numpy(float)
    endpoint_pred = np.full_like(y_true, last_tvt, dtype=float)
    start_pos = max(0, last_pos - 49)
    md0 = float(horizontal["MD"].iloc[start_pos])
    tvt0 = float(horizontal["TVT_input"].iloc[start_pos])
    slope50 = 0.0 if abs(last_md - md0) < 1e-9 else (last_tvt - tvt0) / (last_md - md0)
    slope_pred = last_tvt + slope50 * (md_eval - last_md)
    return {
        "well": well,
        "target_rows": int(target.sum()),
        "endpoint_hold_rmse": rmse_value(y_true, endpoint_pred),
        "endpoint_hold_mae": float(np.mean(np.abs(y_true - endpoint_pred))),
        "slope50_projection_rmse": rmse_value(y_true, slope_pred),
        "slope50_projection_mae": float(np.mean(np.abs(y_true - slope_pred))),
        "last_known_tvt": last_tvt,
        "slope50": slope50,
        "target_delta_mean": float(np.mean(y_true - last_tvt)),
        "target_delta_std": float(np.std(y_true - last_tvt)),
    }

baseline_rows = [per_well_target_baselines(well) for well in sorted(well_descriptors["well"])]
endpoint_baselines = pd.DataFrame([row for row in baseline_rows if row is not None])
endpoint_path = TABLE_DIR / f"eda__phase4_endpoint_baselines_by_well__{RUN_DATE}.csv"
endpoint_baselines.to_csv(endpoint_path, index=False)

print({
    "wells": len(endpoint_baselines),
    "endpoint_row_weighted_rmse": rmse_value(
        np.repeat(0.0, len(endpoint_baselines)),
        np.repeat(0.0, len(endpoint_baselines)),
    ) if False else round(float(np.sqrt(np.average(endpoint_baselines["endpoint_hold_rmse"] ** 2, weights=endpoint_baselines["target_rows"]))), 4),
    "slope50_row_weighted_rmse": round(float(np.sqrt(np.average(endpoint_baselines["slope50_projection_rmse"] ** 2, weights=endpoint_baselines["target_rows"]))), 4),
})
endpoint_baselines.head()


{'wells': 773, 'endpoint_row_weighted_rmse': 15.9099, 'slope50_row_weighted_rmse': 112.4817}


,well,target_rows,endpoint_hold_rmse,endpoint_hold_mae,slope50_projection_rmse,slope50_projection_mae,last_known_tvt,slope50,target_delta_mean,target_delta_std
0,000d7d20,3836,7.454442,6.134966,43.278761,36.768015,11747.37,0.016327,-5.445566,5.090631
1,00bbac68,6014,15.263052,12.104335,54.620503,41.974710,12223.54,0.013061,-2.521854,15.053273
2,00e12e8b,4301,7.924589,6.519877,106.785892,90.312195,11604.82,0.041633,-0.760342,7.888028
3,015fe0d2,4296,23.154890,19.800242,51.839449,43.390020,12233.78,0.029388,19.749576,12.087314
4,01869cd4,5557,10.618593,8.635674,21.116999,18.785098,11745.94,0.008367,4.467760,9.632946


In [4]:
"Build current sorted folds and descriptor-balanced folds."


def make_current_fold_map(wells):
    rows = []
    for fold_idx, (_, valid_wells) in enumerate(group_well_folds(sorted(wells), n_splits=N_SPLITS), start=1):
        for well in valid_wells:
            rows.append({"fold_scheme": "current_sorted", "fold": fold_idx, "well": well})
    return pd.DataFrame(rows)


def make_balanced_fold_map(descriptors, columns):
    work = descriptors[["well"] + columns].copy()
    for column in columns:
        work[column] = pd.to_numeric(work[column], errors="coerce")
        work[column] = work[column].fillna(work[column].median())
    scaled = StandardScaler().fit_transform(work[columns])
    score = np.linalg.norm(scaled, axis=1)
    work["difficulty_score"] = score
    work["eval_rows"] = descriptors["eval_rows"].to_numpy(float)
    fold_totals = {fold: 0.0 for fold in range(1, N_SPLITS + 1)}
    fold_counts = {fold: 0 for fold in range(1, N_SPLITS + 1)}
    rows = []
    for row in work.sort_values(["difficulty_score", "eval_rows"], ascending=False).itertuples(index=False):
        fold = sorted(fold_totals, key=lambda f: (fold_totals[f], fold_counts[f]))[0]
        rows.append({"fold_scheme": "balanced_descriptor", "fold": fold, "well": row.well})
        fold_totals[fold] += float(row.eval_rows)
        fold_counts[fold] += 1
    return pd.DataFrame(rows)

fold_balance_columns = [
    "eval_frac", "eval_rows", "md_range", "gr_missing_frac", "known_tvt_slope_last_50",
    "eval_tvt_delta_from_last_known", "horiz_typewell_gr_corr",
]
current_folds = make_current_fold_map(well_descriptors["well"])
balanced_folds = make_balanced_fold_map(well_descriptors, fold_balance_columns)
fold_map = pd.concat([current_folds, balanced_folds], ignore_index=True)
fold_map_path = TABLE_DIR / f"eda__phase4_fold_assignments__{RUN_DATE}.csv"
fold_map.to_csv(fold_map_path, index=False)

fold_descriptor_frame = fold_map.merge(well_descriptors, on="well", how="left").merge(endpoint_baselines, on="well", how="left")
summary_rows = []
for (scheme, fold), group in fold_descriptor_frame.groupby(["fold_scheme", "fold"]):
    row = {"fold_scheme": scheme, "fold": int(fold), "n_wells": int(group["well"].nunique()), "eval_rows": int(group["eval_rows"].sum())}
    for column in fold_balance_columns:
        row[f"{column}_mean"] = float(group[column].mean())
    row["endpoint_hold_rmse_weighted"] = float(np.sqrt(np.average(group["endpoint_hold_rmse"] ** 2, weights=group["target_rows"])))
    row["slope50_projection_rmse_weighted"] = float(np.sqrt(np.average(group["slope50_projection_rmse"] ** 2, weights=group["target_rows"])))
    summary_rows.append(row)
fold_summary = pd.DataFrame(summary_rows).sort_values(["fold_scheme", "fold"])
fold_summary_path = TABLE_DIR / f"eda__phase4_fold_descriptor_summary__{RUN_DATE}.csv"
fold_summary.to_csv(fold_summary_path, index=False)

fold_variability = []
for scheme, group in fold_summary.groupby("fold_scheme"):
    row = {"fold_scheme": scheme}
    row["eval_rows_cv"] = float(group["eval_rows"].std(ddof=0) / group["eval_rows"].mean())
    row["endpoint_rmse_range"] = float(group["endpoint_hold_rmse_weighted"].max() - group["endpoint_hold_rmse_weighted"].min())
    for column in fold_balance_columns:
        values = group[f"{column}_mean"]
        row[f"{column}_mean_range"] = float(values.max() - values.min())
    fold_variability.append(row)
fold_variability = pd.DataFrame(fold_variability)
fold_variability_path = TABLE_DIR / f"eda__phase4_fold_variability__{RUN_DATE}.csv"
fold_variability.to_csv(fold_variability_path, index=False)
fold_summary


,fold_scheme,fold,n_wells,eval_rows,eval_frac_mean,eval_rows_mean,md_range_mean,gr_missing_frac_mean,known_tvt_slope_last_50_mean,eval_tvt_delta_from_last_known_mean,horiz_typewell_gr_corr_mean,endpoint_hold_rmse_weighted,slope50_projection_rmse_weighted
0,balanced_descriptor,1,253,1263193,0.738372,4992.857708,6679.648221,0.305405,0.003874,0.854269,0.802648,15.712266,120.531490
1,balanced_descriptor,2,261,1260247,0.727545,4828.532567,6550.505747,0.291642,0.001571,2.136475,0.801793,14.747476,118.685633
2,balanced_descriptor,3,259,1260549,0.733730,4866.984556,6532.235521,0.284801,0.000579,-0.048842,0.801209,17.175899,96.636916
3,current_sorted,1,258,1259489,0.732787,4881.740310,6575.891473,0.282131,0.002674,1.649380,0.799945,16.042879,96.003053
4,current_sorted,2,258,1257350,0.733686,4873.449612,6558.453488,0.304226,0.001860,1.475853,0.805952,16.562104,119.642346
5,current_sorted,3,257,1267150,0.733010,4930.544747,6625.762646,0.295210,0.001440,-0.175914,0.799726,15.095777,120.073355


In [5]:
"Collect prior model fold metrics and public leaderboard gaps."

model_metric_files = [
    ("ridge_alpha_10", "linear", TABLE_DIR / "model__linear_ols_ridge_lasso_fold_metrics__2026-06-21__1218__linear_ols_ridge_lasso__14bef53.csv"),
    ("random_forest", "trees", TABLE_DIR / "model__random_forest_metrics__2026-06-21__1310__tree_random_forest__4eb4a55.csv"),
    ("elasticnet_alpha_0p001_l1_0p2", "linear", TABLE_DIR / "model__elasticnet_huber_quantile_fold_metrics__2026-06-21__1412__linear_elasticnet_huber_quantile__448574b.csv"),
]
fold_metric_frames = []
for model_name, model_family, path in model_metric_files:
    metrics = pd.read_csv(path)
    if "model" in metrics.columns:
        metrics = metrics.loc[metrics["model"] == model_name].copy()
    if "candidate" in metrics.columns:
        metrics = metrics.loc[metrics["candidate"] == model_name].copy()
    metrics["model_name"] = model_name
    metrics["model_family"] = model_family
    fold_metric_frames.append(metrics)
model_fold_metrics = pd.concat(fold_metric_frames, ignore_index=True)
if "fold" not in model_fold_metrics.columns:
    model_fold_metrics["fold"] = model_fold_metrics.groupby("model_name").cumcount() + 1
model_fold_metrics = model_fold_metrics[["model_name", "model_family", "fold", "rmse", "mae", "r2"]].sort_values(["model_name", "fold"])
model_fold_metrics_path = TABLE_DIR / f"eda__phase4_model_fold_metrics__{RUN_DATE}.csv"
model_fold_metrics.to_csv(model_fold_metrics_path, index=False)

cv_vs_lb = model_comparison.merge(leaderboard, on="model_name", how="inner")
cv_vs_lb["lb_minus_cv"] = cv_vs_lb["public_lb_rmse"] - cv_vs_lb["cv_rmse_mean"]
cv_vs_lb["abs_lb_minus_cv"] = cv_vs_lb["lb_minus_cv"].abs()
cv_vs_lb = cv_vs_lb.sort_values("public_lb_rmse")
cv_lb_path = TABLE_DIR / f"eda__phase4_cv_leaderboard_gap__{RUN_DATE}.csv"
cv_vs_lb.to_csv(cv_lb_path, index=False)

model_fold_metrics.head(), cv_vs_lb


(                      model_name model_family  fold       rmse        mae  \
 6  elasticnet_alpha_0p001_l1_0p2       linear     1  15.794230  11.278682   
 7  elasticnet_alpha_0p001_l1_0p2       linear     2  16.346409  11.515974   
 8  elasticnet_alpha_0p001_l1_0p2       linear     3  15.231382  10.841049   
 3                  random_forest        trees     1  15.849366  11.384016   
 4                  random_forest        trees     2  16.792813  11.703001   
 
          r2  
 6  0.999343  
 7  0.999363  
 8  0.999383  
 3  0.999338  
 4  0.999328  ,
                                               run_id model_family  \
 1      2026-06-21__1310__tree_random_forest__4eb4a55        trees   
 0  2026-06-21__1218__linear_ols_ridge_lasso__14bef53       linear   
 2  2026-06-21__1412__linear_elasticnet_huber_quan...       linear   
 
                       model_name  cv_rmse_mean  cv_rmse_std  train_time_s  \
 1                  random_forest     16.065506     0.528199     78.204902   
 

In [6]:
"Create validation diagnostic figures."

fig_paths = {}

fig, ax = plt.subplots(figsize=(8, 5))
for model_name, group in model_fold_metrics.groupby("model_name"):
    ax.plot(group["fold"], group["rmse"], marker="o", label=model_name)
ax.set_title("Historical Local CV Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend()
fig.tight_layout()
fig_paths["model_fold_rmse"] = FIGURE_DIR / f"eda__phase4_model_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["model_fold_rmse"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
plot_frame = fold_summary.copy()
for idx, metric in enumerate(["eval_rows", "endpoint_hold_rmse_weighted", "gr_missing_frac_mean"]):
    for scheme, group in plot_frame.groupby("fold_scheme"):
        offset = -0.18 if scheme == "balanced_descriptor" else 0.18
        x = group["fold"].to_numpy(float) + offset + idx * 0.0
        y = group[metric].to_numpy(float)
        y_scaled = y / np.nanmean(y)
        ax.plot(x, y_scaled, marker="o", linestyle="-", label=f"{scheme} {metric}")
ax.set_title("Fold Balance Diagnostics Scaled to Mean")
ax.set_xlabel("fold")
ax.set_ylabel("fold value / mean")
ax.legend(fontsize=8)
fig.tight_layout()
fig_paths["fold_balance"] = FIGURE_DIR / f"eda__phase4_fold_balance__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_balance"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
positions = np.arange(len(cv_vs_lb))
ax.bar(positions - 0.18, cv_vs_lb["cv_rmse_mean"], width=0.36, label="local CV", color="#4C78A8")
ax.bar(positions + 0.18, cv_vs_lb["public_lb_rmse"], width=0.36, label="public LB", color="#F58518")
ax.set_xticks(positions)
ax.set_xticklabels(cv_vs_lb["model_name"], rotation=25, ha="right")
ax.set_title("Local CV vs Public Leaderboard")
ax.set_ylabel("RMSE")
ax.legend()
fig.tight_layout()
fig_paths["cv_vs_leaderboard"] = FIGURE_DIR / f"eda__phase4_cv_vs_leaderboard__{RUN_DATE}.png"
fig.savefig(fig_paths["cv_vs_leaderboard"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(endpoint_baselines["endpoint_hold_rmse"], endpoint_baselines["slope50_projection_rmse"], s=np.clip(endpoint_baselines["target_rows"] / 100, 5, 80), alpha=0.45, color="#54A24B")
limit = np.nanpercentile(endpoint_baselines[["endpoint_hold_rmse", "slope50_projection_rmse"]].to_numpy(float), 95)
ax.plot([0, limit], [0, limit], color="black", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_title("Endpoint Hold vs Direct Slope Projection")
ax.set_xlabel("endpoint-hold RMSE by well")
ax.set_ylabel("last-50 slope projection RMSE by well")
fig.tight_layout()
fig_paths["endpoint_vs_slope"] = FIGURE_DIR / f"eda__phase4_endpoint_vs_slope__{RUN_DATE}.png"
fig.savefig(fig_paths["endpoint_vs_slope"], dpi=160)
plt.close(fig)

pca_columns = ["eval_frac", "md_range", "gr_missing_frac", "known_tvt_slope_last_50", "eval_tvt_delta_from_last_known", "horiz_typewell_gr_corr"]
pca_frame = well_descriptors[["well"] + pca_columns].copy()
for column in pca_columns:
    pca_frame[column] = pd.to_numeric(pca_frame[column], errors="coerce").fillna(pca_frame[column].median())
components = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(pca_frame[pca_columns]))
pca_frame["pc1"] = components[:, 0]
pca_frame["pc2"] = components[:, 1]
pca_frame = pca_frame.merge(current_folds, on="well", how="left")
pca_path = TABLE_DIR / f"eda__phase4_well_descriptor_pca__{RUN_DATE}.csv"
pca_frame.to_csv(pca_path, index=False)
fig, ax = plt.subplots(figsize=(7, 5))
for fold, group in pca_frame.groupby("fold"):
    ax.scatter(group["pc1"], group["pc2"], s=16, alpha=0.65, label=f"fold {fold}")
ax.set_title("Current Fold Coverage in Descriptor PCA Space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
fig.tight_layout()
fig_paths["descriptor_pca"] = FIGURE_DIR / f"eda__phase4_descriptor_pca__{RUN_DATE}.png"
fig.savefig(fig_paths["descriptor_pca"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}


{'model_fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase4_model_fold_rmse__2026-06-21.png',
 'fold_balance': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase4_fold_balance__2026-06-21.png',
 'cv_vs_leaderboard': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase4_cv_vs_leaderboard__2026-06-21.png',
 'endpoint_vs_slope': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase4_endpoint_vs_slope__2026-06-21.png',
 'descriptor_pca': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase4_descriptor_pca__2026-06-21.png'}

In [7]:
"Write findings, report, and run bundle."

current_var = fold_variability.loc[fold_variability["fold_scheme"] == "current_sorted"].iloc[0]
balanced_var = fold_variability.loc[fold_variability["fold_scheme"] == "balanced_descriptor"].iloc[0]
endpoint_weighted = float(np.sqrt(np.average(endpoint_baselines["endpoint_hold_rmse"] ** 2, weights=endpoint_baselines["target_rows"])))
slope_weighted = float(np.sqrt(np.average(endpoint_baselines["slope50_projection_rmse"] ** 2, weights=endpoint_baselines["target_rows"])))
best_lb = cv_vs_lb.iloc[0]
worst_gap = cv_vs_lb.sort_values("abs_lb_minus_cv", ascending=False).iloc[0]
rf_row = cv_vs_lb.loc[cv_vs_lb["model_name"] == "random_forest"].iloc[0]
ridge_row = cv_vs_lb.loc[cv_vs_lb["model_name"] == "ridge_alpha_10"].iloc[0]

findings = pd.DataFrame([
    {
        "topic": "endpoint_baseline",
        "finding": f"Full-row endpoint-hold RMSE is {endpoint_weighted:.4f}; direct last-50 slope projection RMSE is {slope_weighted:.4f}.",
        "modeling_implication": "Endpoint hold should be included as a validation baseline. Direct recent-slope extrapolation should not be used as a standalone model.",
    },
    {
        "topic": "fold_balance",
        "finding": f"Current sorted folds have eval-row CV {current_var.eval_rows_cv:.3f} and endpoint-RMSE range {current_var.endpoint_rmse_range:.3f}; balanced descriptor folds have eval-row CV {balanced_var.eval_rows_cv:.3f} and endpoint-RMSE range {balanced_var.endpoint_rmse_range:.3f}.",
        "modeling_implication": "Future CV should use descriptor-balanced grouped folds or at least report descriptor balance; sorted grouped folds are acceptable only as a baseline diagnostic.",
    },
    {
        "topic": "local_vs_public_mismatch",
        "finding": f"Random Forest public LB beats Ridge by {ridge_row.public_lb_rmse - rf_row.public_lb_rmse:.3f}, while local CV ranks Random Forest worse than Ridge by {rf_row.cv_rmse_mean - ridge_row.cv_rmse_mean:.3f}.",
        "modeling_implication": "Treat the Random Forest public gain as tentative until the validation split is refined and repeated on richer features.",
    },
    {
        "topic": "leaderboard_gap",
        "finding": f"Largest absolute public-minus-CV gap is {worst_gap.abs_lb_minus_cv:.3f} for {worst_gap.model_name}.",
        "modeling_implication": "Avoid leaderboard-only tuning; use public LB as a coarse external check after local diagnostics agree.",
    },
    {
        "topic": "recommended_protocol",
        "finding": "Use grouped well-level folds, descriptor balance reporting, endpoint-hold baseline, and model fold-level error tables for each reportable modeling notebook.",
        "modeling_implication": "The next LightGBM/XGBoost pass should compare against Ridge, Random Forest, endpoint hold, and balanced-fold diagnostics before Kaggle submission.",
    },
])

findings_path = TABLE_DIR / f"eda__phase4_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"eda__phase4_validation__{RUN_DATE}.md"
report_lines = [
    "# Phase 4 Validation EDA",
    "",
    f"Date: {RUN_DATE}",
    "",
    "## Key Findings",
]
for row in findings.itertuples(index=False):
    report_lines.extend([
        f"- **{row.topic}**: {row.finding}",
        f"  - Modeling implication: {row.modeling_implication}",
    ])
report_lines.extend([
    "",
    "## Artifacts",
    f"- Endpoint baselines by well: {endpoint_path}",
    f"- Fold assignments: {fold_map_path}",
    f"- Fold descriptor summary: {fold_summary_path}",
    f"- Fold variability: {fold_variability_path}",
    f"- Model fold metrics: {model_fold_metrics_path}",
    f"- CV / leaderboard gap: {cv_lb_path}",
    f"- Descriptor PCA table: {pca_path}",
    f"- Findings table: {findings_path}",
])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

config = {
    "task": "phase4_validation_eda",
    "seed": SEED,
    "n_splits": N_SPLITS,
    "fold_schemes": ["current_sorted", "balanced_descriptor"],
    "note": "Validation diagnostics only; no Kaggle submission model trained.",
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_input(str(TABLE_DIR / f"eda__phase2_train_well_summary__{RUN_DATE}.csv"))
    rb.add_input(str(TABLE_DIR / f"eda__phase2_typewell_alignment__{RUN_DATE}.csv"))
    rb.add_metric("train_wells", float(len(well_descriptors)))
    rb.add_metric("endpoint_hold_rmse", endpoint_weighted)
    rb.add_metric("slope50_projection_rmse", slope_weighted)
    rb.add_metric("current_eval_rows_cv", float(current_var.eval_rows_cv))
    rb.add_metric("balanced_eval_rows_cv", float(balanced_var.eval_rows_cv))
    rb.add_metric("random_forest_public_minus_cv", float(rf_row.lb_minus_cv))
    rb.add_metric("ridge_public_minus_cv", float(ridge_row.lb_minus_cv))
    for output_path in [endpoint_path, fold_map_path, fold_summary_path, fold_variability_path, model_fold_metrics_path, cv_lb_path, pca_path, findings_path, report_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("Phase 4 validation EDA completed. Recommends descriptor-balanced grouped folds plus endpoint-hold baseline reporting before boosted-tree submissions.")
run_id = rb.run_id

print({
    "run_id": run_id,
    "endpoint_hold_rmse": round(endpoint_weighted, 4),
    "slope50_projection_rmse": round(slope_weighted, 4),
    "current_eval_rows_cv": round(float(current_var.eval_rows_cv), 4),
    "balanced_eval_rows_cv": round(float(balanced_var.eval_rows_cv), 4),
})
findings


{'run_id': '2026-06-21__1928__eda_phase4_validation__203566d', 'endpoint_hold_rmse': 15.9099, 'slope50_projection_rmse': 112.4817, 'current_eval_rows_cv': 0.0033, 'balanced_eval_rows_cv': 0.001}


,topic,finding,modeling_implication
0,endpoint_baseline,Full-row endpoint-hold RMSE is 15.9099; direct...,Endpoint hold should be included as a validati...
1,fold_balance,Current sorted folds have eval-row CV 0.003 an...,Future CV should use descriptor-balanced group...
2,local_vs_public_mismatch,"Random Forest public LB beats Ridge by 0.032, ...",Treat the Random Forest public gain as tentati...
3,leaderboard_gap,Largest absolute public-minus-CV gap is 0.300 ...,Avoid leaderboard-only tuning; use public LB a...
4,recommended_protocol,"Use grouped well-level folds, descriptor balan...",The next LightGBM/XGBoost pass should compare ...


## Results

The executed notebook exports fold assignment diagnostics, descriptor summaries, endpoint/slope baseline tables, model fold metrics, CV/leaderboard gaps, figures, a markdown report, and a run bundle.

## Takeaways

Phase 4 should control the next modeling loop: compare every richer model against endpoint hold, Ridge, and Random Forest under grouped well-level validation, and report fold descriptor balance before using public leaderboard feedback.
